# ===== Scenario: 5-class =====

In [ ]:
from utils import *
(x_train, y_train, x_val, y_val, x_test, y_test, _, _, _) = load_jetnet(
    mode="5class",
    n_train_pool=620000,
    n_test_pool=260000,
    n_train=250000,
    n_val=100000,
    n_test=250000,
    seed=42
)

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(20, 4))
class_names = ['q', 'g', 'W', 'Z', 't']

for i, class_name in enumerate(class_names):
    class_idx = np.where(y_train.argmax(axis=1)==i)[0][0]
    jet = x_train[class_idx]
    eta = jet[:,0]
    phi = jet[:,1]
    pt = jet[:,2]
    mask = jet[:,3]
    
    eta = eta[mask==1]
    phi = phi[mask==1]
    pt = pt[mask==1]

    size = pt*10000

    axs[i].scatter(eta, phi, s=size, facecolors="none", edgecolors="C0", linewidths=0.8, alpha=0.5)
    axs[i].set_xlabel("Eta")
    axs[i].set_ylabel("Phi")
    axs[i].set_title(class_name)
    axs[i].set_xlim(-0.4, 0.4)
    axs[i].set_ylim(-0.4, 0.4)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
class_names = ['q', 'g', 'W', 'Z', 't']

ratio_range = (0.1, 0.5)
lw = 0.8
alpha = 0.5
for col, cname in enumerate(class_names):
    class_idxs = np.where(y_train.argmax(axis=1) == col)[0]
    idx = class_idxs[0] # example idx
    jet_orig = x_train[idx]

    # row 0: original
    eta, phi, pt, valid = jet_orig.T
    m0 = (valid == 1)
    axes[0, col].scatter(eta[m0], phi[m0], s=pt[m0]*1e4, facecolors="none", edgecolors="C0", linewidths=lw, alpha=alpha)
    axes[0, col].set_title(f"{cname} [original]")

    # row 1: augmented
    jet_aug = augment_jet(jet_orig)
    eta, phi, pt, valid = jet_aug.T
    m1 = (valid == 1)
    axes[1, col].scatter(eta[m1], phi[m1], s=pt[m1]*1e4, facecolors="none", edgecolors="C0", linewidths=lw, alpha=alpha)
    axes[1, col].set_title(f"{cname} [augmented]")

    # row 2: augmented + masks
    r_target = float(RNG.uniform(ratio_range[0], ratio_range[1]))
    jet_tf = tf.convert_to_tensor(jet_aug[None, ...], dtype=tf.float32)
    mask_bool = tf_make_masks_pt_ratio(jet_tf, p_mask=1, ratio_range=(r_target, r_target)).numpy()[0].astype(bool)

    total_pt = float(pt[m1].sum())
    masked_pt = float(pt[mask_bool].sum())
    r_ach = masked_pt / total_pt

    unmasked = (valid == 1) & (~mask_bool)
    masked = (valid == 1) & (mask_bool)

    axes[2, col].scatter(eta[unmasked], phi[unmasked], s=pt[unmasked]*1e4, facecolors="none", edgecolors="C0", linewidths=lw, alpha=alpha)
    axes[2, col].scatter(eta[masked], phi[masked], s=pt[masked]*1e4, facecolors="none", edgecolors="red", linewidths=lw, alpha=alpha)

    axes[2, col].set_title(f"{cname} [augmented+masked]")

    print(f"{cname}: r_target={r_target:.3f}, r_achieved={r_ach:.3f}, n_mask={masked.sum()} / n_valid={m1.sum()}")

    for row in range(3):
        axes[row, col].set_xlim(-0.4, 0.4)
        axes[row, col].set_ylim(-0.4, 0.4)
        axes[row, col].set_xlabel("Eta")
        axes[row, col].set_ylabel("Phi")

plt.tight_layout()
plt.show()

### pretraining

In [ ]:
d_model=64
n_heads=4
n_layers=4
d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")

proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

teacher.set_weights(student.get_weights())
proj_head_t.set_weights(proj_head_s.get_weights())
teacher.trainable = False
proj_head_t.trainable = False

masker = MaskTokens(d_model, name="particle_masker")
center_cls = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_cls")
center_patch = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_patch")

print(student.summary())
print(proj_head_s.summary())

batch_size = 1024
base_lr = 5e-4 * (batch_size / 256)

history_jbot_all = train_jbot(
    x_train=x_train,
    epochs=80,
    batch_size=batch_size,
    optimizer=keras.optimizers.AdamW(learning_rate=base_lr, weight_decay=1e-5),
    base_lr=base_lr,
    warmup_epochs=10,
    ema_tau=0.996,
    student=student,
    teacher=teacher,
    n_layers=n_layers,
    d_proj=d_proj,
    proj_head_s=proj_head_s,
    proj_head_t=proj_head_t,
    mask_prob=1,
    mask_ratio_range=(0.1, 0.5),
    masker=masker,
    center_cls=center_cls,
    center_patch=center_patch,
    center_beta=0.9,
    temp_t=0.04,
    temp_s=0.1,
    use_hint=False,
    hint_hidden=0,
    lambda_koleo=0.05)

plot_jbot_training(history_jbot_all)

In [ ]:
plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_train, y_sample=y_train)
plot_softmax_prob_cls(n_samples=5000, x=x_train,
                      student=student, teacher=teacher,
                      proj_head_s=proj_head_s, proj_head_t=proj_head_t,
                      center_cls=center_cls, temp_t=0.04, temp_s=0.1)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4)

In [ ]:
save_jbot = False
#save_jbot = True
if save_jbot:
    student.save_weights("models/all_jbot_back_s.weights.h5")
    teacher.save_weights("models/all_jbot_back_t.weights.h5")
    proj_head_s.save_weights("models/all_jbot_proj_s.weights.h5")
    proj_head_t.save_weights("models/all_jbot_proj_t.weights.h5")

### downstream

In [ ]:
d_model=64
n_heads=4
n_layers=4
d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")
proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

student.load_weights("models/all_jbot_back_s.weights.h5")
teacher.load_weights("models/all_jbot_back_t.weights.h5")
proj_head_s.load_weights("models/all_jbot_proj_s.weights.h5")
proj_head_t.load_weights("models/all_jbot_proj_t.weights.h5")

In [ ]:
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4)

In [ ]:
z_cls_train = student.predict(x_train)[0]
z_cls_test = student.predict(x_test)[0]

y_train_idx = y_train.argmax(1)
y_test_idx = y_test.argmax(1)

knn = KNeighborsClassifier(n_neighbors=20).fit(z_cls_train, y_train_idx)
acc_knn = accuracy_score(y_test_idx, knn.predict(z_cls_test))

scaler = StandardScaler().fit(z_cls_train)
Z_cls_train = scaler.transform(z_cls_train)
Z_cls_test = scaler.transform(z_cls_test)
logreg = LogisticRegression(max_iter=1000).fit(Z_cls_train, y_train_idx)
acc_linear = accuracy_score(y_test_idx, logreg.predict(Z_cls_test))

print(f"k-NN acc:   {acc_knn:.3f}")
print(f"linear acc: {acc_linear:.3f}")

knn_p = knn.predict_proba(z_cls_test)
linear_p = logreg.predict_proba(Z_cls_test)

plt.figure(figsize=(7,6))
class_names = ['q', 'g', 'W', 'Z', 't']
class_colors = {'q': 'C3', 'g': 'C1', 'W': 'C2', 'Z': 'C0', 't': 'C4'}

for k, c in enumerate(class_names):
    fpr_knn, tpr_knn, _ = roc_curve(y_test[:, k], knn_p[:, k])
    auc_knn = auc(fpr_knn, tpr_knn)
    plt.plot(tpr_knn, fpr_knn, color=class_colors[c], label=f"{c} [k-NN] ({auc_knn:.3f})", linestyle="-", lw=1.5)

    fpr_linear, tpr_linear, _ = roc_curve(y_test[:, k], linear_p[:, k])
    auc_linear = auc(fpr_linear, tpr_linear)
    plt.plot(tpr_linear, fpr_linear, color=class_colors[c], label=f"{c} [linear] ({auc_linear:.3f})", linestyle="--", lw=1.5)

plt.xlabel("TPR", size=16)
plt.ylabel("FPR", size=16)
plt.yscale("log")
plt.ylim(1e-4, 1)
plt.title("k-NN/linear probe on [CLS] embedding", size=15)
plt.legend(fontsize=10, loc='lower right')
plt.show()

In [ ]:
epochs = 200
batch_size = 1024
lr = 1e-4 * (batch_size / 256)
tolerance = 1e-4
patience = 15

backbone_frozen = build_backbone(d_model, n_heads, n_layers, name="backbone_frozen")
backbone_frozen.load_weights("models/jbot_back_s.weights.h5")
backbone_frozen.trainable = False
mlp_frozen = build_mlp(d_model, name="mlp_frozen")
x_in = tf.keras.Input((30,4))
cls, _ = backbone_frozen(x_in)
x_out = mlp_frozen(cls)
model_frozen = tf.keras.models.Model(x_in, x_out, name="model_frozen")
model_frozen.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])

backbone_standalone = build_backbone(d_model, n_heads, n_layers, name="backbone_standalone")
mlp_standalone = build_mlp(d_model, name="mlp_standalone")
x_in = tf.keras.Input((30,4))
cls, _ = backbone_standalone(x_in)
x_out = mlp_standalone(cls)
model_standalone = tf.keras.models.Model(x_in, x_out, name="model_standalone")
model_standalone.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])

backbone_ft = build_backbone(d_model, n_heads, n_layers, name="backbone_ft")
backbone_ft.load_weights("models/jbot_back_s.weights.h5")
mlp_ft = build_mlp(d_model, name="mlp_ft")


history_frozen = model_frozen.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=epochs, batch_size=batch_size, verbose=0, callbacks=[EarlyStoppingLogger("jBOT (frozen)",epochs,patience,tolerance)])
history_standalone = model_standalone.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=epochs, batch_size=batch_size, verbose=0, callbacks=[EarlyStoppingLogger("supervised",epochs,patience,tolerance)])
model_ft, history_ft = finetune_llrd(
    backbone=backbone_ft,
    mlp=mlp_ft,
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    n_layers=n_layers,
    base_lr=lr,
    decay=0.7,
    epochs=epochs,
    batch_size=batch_size,
    tolerance=tolerance,
    patience=patience
)

plot_training_histories(history_frozen, history_standalone, history_ft, labels=("jBOT (frozen)", "supervised", "jBOT"))

In [ ]:
y_pred_frozen = model_frozen.predict(x_test)
y_pred_standalone = model_standalone.predict(x_test)
y_pred_ft = model_ft.predict(x_test)

report_acc(y_test, y_pred_frozen, model_name="jBOT (frozen)", k_folds=10)
report_acc(y_test, y_pred_standalone, model_name="supervised", k_folds=10)
report_acc(y_test, y_pred_ft, model_name="jBOT", k_folds=10)

plt.figure(figsize=(7,6))
class_names = ['q','g','W','Z','t']
class_colors = ['C3','C1','C2','C0','C4']

for k,c in enumerate(class_names):
    fpr_standalone, tpr_standalone, _ = roc_curve(y_test[:,k], y_pred_standalone[:,k])
    auc_standalone = auc(fpr_standalone, tpr_standalone)
    plt.plot(tpr_standalone, fpr_standalone, color=class_colors[k], lw=1.5, linestyle='-', label=f"{c} [Supervised] ({auc_standalone:.3f})")

    fpr_ft, tpr_ft, _ = roc_curve(y_test[:,k], y_pred_ft[:,k])
    auc_ft = auc(fpr_ft, tpr_ft)
    plt.plot(tpr_ft, fpr_ft, color=class_colors[k], lw=1.5, linestyle='dashed', label=f"{c} [jBOT] ({auc_ft:.3f})")

plt.xlabel("TPR", size=16)
plt.ylabel("FPR", size=16)
plt.yscale("log")
plt.ylim(1e-4,1)
plt.title("MLP head on backbone CLS embedding", size=16)
plt.legend(fontsize=9)
plt.show()

In [ ]:
plot_tSNE_cls(n_samples=5000, backbone=backbone_ft, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=backbone_ft, x=x_test, y=y_test, n_samples=5000, n_components=4)

# ===== Scenario: top vs qg =====

In [ ]:
from utils import *
(x_train, y_train, x_val, y_val, x_test, y_test, y_train_top, y_val_top, y_test_top) = load_jetnet(
    mode="tqg",
    n_train_pool=620000,
    n_test_pool=260000,
    n_train=200000,
    n_val=50000,
    n_test=100000,
    seed=42
)

### pretraining

In [ ]:
d_model=64
n_heads=4
n_layers=4
d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")

proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

teacher.set_weights(student.get_weights())
proj_head_t.set_weights(proj_head_s.get_weights())
teacher.trainable = False
proj_head_t.trainable = False

masker = MaskTokens(d_model, name="particle_masker")
center_cls = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_cls")
center_patch = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_patch")

print(student.summary())
print(proj_head_s.summary())

batch_size = 1024
base_lr = 5e-4 * (batch_size / 256)

history_jbot_all = train_jbot(
    x_train=x_train,
    epochs=100,
    batch_size=batch_size,
    optimizer=keras.optimizers.AdamW(learning_rate=base_lr, weight_decay=1e-5),
    base_lr=base_lr,
    warmup_epochs=10,
    ema_tau=0.996,
    student=student,
    teacher=teacher,
    n_layers=n_layers,
    d_proj=d_proj,
    proj_head_s=proj_head_s,
    proj_head_t=proj_head_t,
    mask_prob=1,
    mask_ratio_range=(0.1, 0.5),
    masker=masker,
    center_cls=center_cls,
    center_patch=center_patch,
    center_beta=0.9,
    temp_t=0.04,
    temp_s=0.1,
    use_hint=False,
    hint_hidden=0,
    lambda_koleo=0.05)

plot_jbot_training(history_jbot_all)

In [ ]:
plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_train, y_sample=y_train)
plot_softmax_prob_cls(n_samples=5000, x=x_train,
                      student=student, teacher=teacher,
                      proj_head_s=proj_head_s, proj_head_t=proj_head_t,
                      center_cls=center_cls, temp_t=0.04, temp_s=0.1)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4)

In [ ]:
#save_jbot = False
save_jbot = True
if save_jbot:
    student.save_weights("models/tqg_jbot_back_s.weights.h5")
    teacher.save_weights("models/tqg_jbot_back_t.weights.h5")
    proj_head_s.save_weights("models/tqg_jbot_proj_s.weights.h5")
    proj_head_t.save_weights("models/tqg_jbot_proj_t.weights.h5")

### downstream (tqg pretraining)

In [ ]:
d_model=64
n_heads=4
n_layers=4
d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")
proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

student.load_weights("models/tqg_jbot_back_s.weights.h5")
teacher.load_weights("models/tqg_jbot_back_t.weights.h5")
proj_head_s.load_weights("models/tqg_jbot_proj_s.weights.h5")
proj_head_t.load_weights("models/tqg_jbot_proj_t.weights.h5")

In [ ]:
plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_train, y_sample=y_train)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4)

### downstream (5-class pretraining)

In [ ]:
d_model=64
n_heads=4
n_layers=4
d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")
proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

student.load_weights("models/all_jbot_back_s.weights.h5")
teacher.load_weights("models/all_jbot_back_t.weights.h5")
proj_head_s.load_weights("models/all_jbot_proj_s.weights.h5")
proj_head_t.load_weights("models/all_jbot_proj_t.weights.h5")

In [ ]:
plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_train, y_sample=y_train)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4)

# ===== Scenario: anomaly detection =====

In [ ]:
from utils import *
(x_train, y_train, _, _, x_test, y_test, _, _, _) = load_jetnet(
    mode="qg",
    n_train=240000,
    qg_balance_anoms=True,
    seed=42
)

### pretraining

In [ ]:
d_model=64
n_heads=4
n_layers=4
d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")

proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

teacher.set_weights(student.get_weights())
proj_head_t.set_weights(proj_head_s.get_weights())
teacher.trainable = False
proj_head_t.trainable = False

masker = MaskTokens(d_model, name="particle_masker")
center_cls = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_cls")
center_patch = tf.Variable(tf.zeros([d_proj]), trainable=False, name="center_patch")

print(student.summary())
print(proj_head_s.summary())

batch_size = 1024
base_lr = 5e-4 * (batch_size / 256)

history_jbot_all = train_jbot(
    x_train=x_train,
    epochs=100,
    batch_size=batch_size,
    optimizer=keras.optimizers.AdamW(learning_rate=base_lr, weight_decay=1e-5),
    base_lr=base_lr,
    warmup_epochs=10,
    ema_tau=0.996,
    student=student,
    teacher=teacher,
    n_layers=n_layers,
    d_proj=d_proj,
    proj_head_s=proj_head_s,
    proj_head_t=proj_head_t,
    mask_prob=1,
    mask_ratio_range=(0.1, 0.5),
    masker=masker,
    center_cls=center_cls,
    center_patch=center_patch,
    center_beta=0.9,
    temp_t=0.04,
    temp_s=0.1,
    use_hint=False,
    hint_hidden=0,
    lambda_koleo=0.05)

plot_jbot_training(history_jbot_all)

In [ ]:
plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_test, y_sample=y_test)
plot_softmax_prob_cls(n_samples=5000, x=x_train,
                      student=student, teacher=teacher,
                      proj_head_s=proj_head_s, proj_head_t=proj_head_t,
                      center_cls=center_cls, temp_t=0.04, temp_s=0.1)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4)

In [ ]:
#save_jbot = False
save_jbot = True
if save_jbot:
    student.save_weights("models/qg_jbot_back_s.weights.h5")
    teacher.save_weights("models/qg_jbot_back_t.weights.h5")
    proj_head_s.save_weights("models/qg_jbot_proj_s.weights.h5")
    proj_head_t.save_weights("models/qg_jbot_proj_t.weights.h5")

### downstream

In [ ]:
d_model=64
n_heads=4
n_layers=4
d_proj=32

student = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_student")
teacher = build_backbone(d_model=d_model, n_heads=n_heads, n_layers=n_layers, name="backbone_teacher")
proj_head_s = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_student")
proj_head_t = build_proj_head(d_in=d_model, d_proj=d_proj, name="proj_head_teacher")

student.load_weights("models/qg_jbot_back_s.weights.h5")
teacher.load_weights("models/qg_jbot_back_t.weights.h5")
proj_head_s.load_weights("models/qg_jbot_proj_s.weights.h5")
proj_head_t.load_weights("models/qg_jbot_proj_t.weights.h5")

In [ ]:
plot_attention(transformer=student, n_heads=n_heads, n_layers=n_layers, x_sample=x_test, y_sample=y_test)
plot_tSNE_cls(n_samples=5000, backbone=student, x=x_test, y=y_test)
plot_pca_corner_cls_embeddings(backbone=student, x=x_test, y=y_test, n_samples=5000, n_components=4)